In [ ]:
!pip install patchify
!pip install segmentation-models-pytorch torchmetrics
# !pip install rasterio

In [ ]:
import os # it is used to interact with virtual machine's file system.
import cv2 # OpenCV for image processing
from PIL import Image # image class from PILLOW library is used to read and manipulate the images using python code
import numpy as np
from patchify import patchify # it is used to split large images into patches
from sklearn.preprocessing import MinMaxScaler, StandardScaler # MinMaxScaler is used to scale an image to a range and StandardScaler removes mean and scales to unit variance
from matplotlib import pyplot as plt # it is used to plot the predictions
import random

In [ ]:
minmaxscaler=MinMaxScaler()

In [ ]:
!ls -lah '/content/drive/MyDrive/Colab Notebooks/Satellite-Data/Dubai-Dataset'

In [ ]:
dataset_root_folder="/content/drive/MyDrive/Colab Notebooks/Satellite-Data/"

In [ ]:
dataset_name="Dubai-Dataset"

In [ ]:
for path , subdirs , files in os.walk(dataset_root_folder):
  dir_name = path.split(os.path.sep)[-1]
  print(dir_name)
  if dir_name == 'images':
    images=os.listdir(path)
    print(images)

In [ ]:
image=cv2.imread(f'{dataset_root_folder}/{dataset_name}/Tile 1/images/image_part_001.jpg',1) # reading the image from the path

In [ ]:
type(Image.fromarray(image))

In [ ]:
image.shape

In [ ]:
image_patch_size=256

In [ ]:
(image.shape[1]//image_patch_size)*image_patch_size

In [ ]:
image_patches=patchify(image,(image_patch_size,image_patch_size,3), step=image_patch_size)

In [ ]:
print(image_patches.shape)

In [ ]:
image_x=image_patches[0,0,:,:]

In [ ]:
#MinMaxScalaer()
image_y=minmaxscaler.fit_transform(image_x.reshape(-1,image_x.shape[-1])).reshape(image_x.shape) #firstly the numpy calculates the no. of rows required then the code after comma flatens the image into 2D array where row->pixel and column->color channel
                                                                                                 #then fits the scalar to the data and transforms the data by scaling all pixel values to be within 0 to 1
                                                                                                 #then after scaling the data is converted back to its original image shape.

print(image_y.shape)

In [ ]:
len(image_patches)

In [ ]:
image_dataset=[]
mask_dataset=[]
for image_type in ['images','masks']:
  if image_type=='images':
    image_extension='jpg'
  elif image_type=='masks':
    image_extension='png'
  for tile_id in range(1,8):
    for image_id in range(1,10):
      image=cv2.imread(f'{dataset_root_folder}{dataset_name}/Tile {tile_id}/{image_type}/image_part_00{image_id}.{image_extension}',1)
      if image is not None:
        if image_type=='masks':
          image=cv2.cvtColor(image,cv2.COLOR_BGR2RGB) #Converting image from BGR to RGB using cvtColor because matplotlib rads the image in RGB
        # print(image.shape)
        size_x=(image.shape[1]//image_patch_size)*image_patch_size  #making image the multiple of patch size
        size_y=(image.shape[0]//image_patch_size)*image_patch_size
        image=Image.fromarray(image)            # converting image from numpy array to PIL
        image=image.crop((0,0,size_x,size_y))   # cropping the image in 256x256
        # print("({} , {})".format(image.size[0],image.size[1]))
        image=np.array(image)                   #converting the image back to numpy array because patchhify takes numpy array as input.
        patched_images=patchify(image,(image_patch_size,image_patch_size,3), step=image_patch_size)
        # print(len(patched_images))
        for i in range(patched_images.shape[0]):
          for j in range (patched_images.shape[1]):

            individual_patched_image=patched_images[i,j,0,:,:,:] # Accessing the patch from the patchify output
              # print(individual_patched_image.shape)
            if image_type=='images':

              individual_patched_image=minmaxscaler.fit_transform(individual_patched_image.reshape(-1,individual_patched_image.shape[-1])).reshape(individual_patched_image.shape)
              # firstly it grabs the no. of colour channel then
              # flattens the image to 2D array then
              # minmaxscaler scales the image between 0 and 1
              # then puts the image back together
              image_dataset.append(individual_patched_image)
            elif image_type=='masks':
              individual_patched_mask=patched_images[i,j,0,:,:,:]
              mask_dataset.append(individual_patched_mask)

In [ ]:
print(len(image_dataset))
print(len(mask_dataset))

In [ ]:
image_dataset=np.array(image_dataset)
mask_dataset=np.array(mask_dataset)

In [ ]:
type(image_dataset[0])

In [ ]:
plt.figure(figsize=(10,8))
plt.subplot(121)
plt.imshow(image_dataset[0])
plt.subplot(122)
plt.imshow(mask_dataset[0])

In [ ]:
class_building ='#3c1098'
class_building=class_building.lstrip('#')
class_building=np.array(tuple(int(class_building[i:i+2],16)for i in (0,2,4))) # gives a tuple of int values of hexadecimal color codes of "building" at indexes 0,2,4.
print(class_building)

In [ ]:
class_building ='#3c1098'
class_building=class_building.lstrip('#')
class_building=np.array(tuple(int(class_building[i:i+2],16)for i in (0,2,4)))
print(class_building)

class_land ='#8429F6'
class_land=class_land.lstrip('#')
class_land=np.array(tuple(int(class_land[i:i+2],16)for i in (0,2,4)))
print(class_land)

class_road ='#6EC1E4'
class_road=class_road.lstrip('#')
class_road=np.array(tuple(int(class_road[i:i+2],16)for i in (0,2,4)))
print(class_road)

class_vegetation ='#FEDD3A'
class_vegetation=class_vegetation.lstrip('#')
class_vegetation=np.array(tuple(int(class_vegetation[i:i+2],16)for i in (0,2,4)))
print(class_vegetation)

class_water ='#E2A929'
class_water=class_water.lstrip('#')
class_water=np.array(tuple(int(class_water[i:i+2],16)for i in (0,2,4)))
print(class_water)

class_unlabelled ='#9B9B9B'
class_unlabelled=class_unlabelled.lstrip('#')
class_unlabelled=np.array(tuple(int(class_unlabelled[i:i+2],16)for i in (0,2,4)))
print(class_unlabelled)

In [ ]:
def rgb_to_label(label):
  label_segment=np.zeros(label.shape, dtype=np.uint8)
  label_segment[np.all(label==class_water, axis=-1)]=0
  label_segment[np.all(label==class_land, axis=-1)]=1
  label_segment[np.all(label==class_road, axis=-1)]=2
  label_segment[np.all(label==class_building, axis=-1)]=3
  label_segment[np.all(label==class_vegetation, axis=-1)]=4
  label_segment[np.all(label==class_unlabelled, axis=-1)]=5
  label_segment=label_segment[:,:,0]
  print(label_segment)
  return label_segment

In [ ]:
labels=[]
for i in range(mask_dataset.shape[0]):
  label=rgb_to_label(mask_dataset[i])
  labels.append(label)

In [ ]:
print(len(labels))

In [ ]:
labels=np.array(labels)

In [ ]:
labels[3]

In [ ]:
master_training_dataset=image_dataset

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(master_training_dataset, labels, test_size=0.2, random_state=42)
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
image_height=X_train.shape[1]
image_width=X_train.shape[2]
image_channel=X_train.shape[3]
print(image_height, image_width, image_channel)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class SatelliteDataset(Dataset):
    def __init__(self, images, masks):
        self.images = images
        self.masks = masks

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx]
        mask = self.masks[idx]

        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) # converting the shape of tensors as pytorch is channel first (3,256,256)
                                                                             # using permute to do so.

        mask_tensor = torch.tensor(mask, dtype=torch.long).unsqueeze(0)

        # Normalize images to values between 0 and 1 (if they are 0-255)
        if img_tensor.max() > 1.0:
            img_tensor = img_tensor / 255.0

        return img_tensor, mask_tensor

train_dataset = SatelliteDataset(X_train, y_train)
test_dataset = SatelliteDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True) # Using GPU
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

print(f"Batches per training epoch: {len(train_loader)}")

In [ ]:
import segmentation_models_pytorch as smp
from segmentation_models_pytorch.losses import DiceLoss, FocalLoss
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = smp.Unet(                   # building the u-net model.
    encoder_name="resnet34",
    encoder_weights="imagenet",
    in_channels=3,
    classes=6,                      # 6 output channels for 6 classes
)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)  # Definiing the optimizer

dice_fn = DiceLoss(mode='multiclass')    # Defining the loss function
focal_fn = FocalLoss(mode='multiclass')

In [ ]:
epochs = 65

for epoch in range(epochs):
    model.train() # Setting model to training mode
    train_loss = 0.0

    for images, masks in train_loader:
        images = images.to(device)  # Move data to GPU

        masks = masks.to(device).long().squeeze(1)  # changing the shape from (16, 1, 256, 256) to (16, 256, 256)

        optimizer.zero_grad()   # gradients

        predictions = model(images)   # Forward pass (making predictions)

        loss = dice_fn(predictions, masks) + focal_fn(predictions, masks)  # calculating the loss

        loss.backward() # back propagation

        optimizer.step()  # Updating the weights

        train_loss =train_loss + loss.item()

    avg_loss = train_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{epochs}] - Combined Dice/Focal Loss: {avg_loss:.4f}")

In [ ]:
from torchmetrics.classification import MulticlassJaccardIndex

# insertion over union (iou) is also known as jaccard index.
iou_metric_per_class = MulticlassJaccardIndex(num_classes=6, average='none').to(device)

model.eval()

with torch.no_grad():
    for images, masks in test_loader:
        images = images.to(device)
        masks = masks.to(device).long().squeeze(1)

        # predictions
        raw_predictions = model(images)
        prob_predictions = torch.softmax(raw_predictions, dim=1)
        pred_classes = torch.argmax(prob_predictions, dim=1)

        iou_metric_per_class.update(pred_classes, masks)  # .update() feeds the batch to torchmetrics.

per_class_iou = iou_metric_per_class.compute()  #.compute() calculates the exact mathematical IoU

# Create a list of your Dubai classes so the output is easy to read
class_names = ["Unlabeled", "Building", "Land", "Road", "Vegetation", "Water"]

print("Per-Class IoU Scores ")
for name, score in zip(class_names, per_class_iou):
    print(f"{name}: {score:.4f}")

print(f"\nOverall Average (mIoU): {per_class_iou.mean():.4f}")  # calculating the overall average (Mean IoU or mIoU)

iou_metric_per_class.reset()  # Clearing the memory for next test

In [ ]:
# Saving the model.
# torch.save(model.state_dict(), 'dubai_6class_model_final_unet.pth')

# print("Model saved successfully.")

In [ ]:
import torch
import segmentation_models_pytorch as smp

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = smp.Unet(           # building the 6-class architecture
    encoder_name="resnet34",
    encoder_weights=None, # None, because we are using the model weights
    in_channels=3,
    classes=6,
)

# lodint the model
model.load_state_dict(torch.load('dubai_6class_model_unet.pth', map_location=device)) # (map_location=device) loads the model correctly whether we are on GPU or cpu
model = model.to(device)

# 3. Lock it for testing
model.eval()
print("6-Class Dubai Model successfully loaded")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch

# Defining the exact Dubai hex colors as RGB arrays to put in matplotlib
color_map = np.array([
    [155, 155, 155], # 0: Unlabeled (#9B9B9B)
    [60,  16,  152], # 1: Building (#3C1098)
    [132, 41,  246], # 2: Land (#8429F6)
    [110, 193, 228], # 3: Road (#6EC1E4)
    [254, 221, 58],  # 4: Vegetation (#FEDD3A)
    [226, 169, 41]   # 5: Water (#E2A929)
], dtype=np.uint8)

# taking just one batch of images from the existing test_loader
model.eval()
images, true_masks = next(iter(test_loader))

# taking the first image in the batch
single_image = images[7].unsqueeze(0).to(device)

# check that the dimensions of mask is (256, 256)
single_mask = true_masks[7].squeeze().numpy()

# putting the image into the model
with torch.no_grad():
    raw_pred = model(single_image)
    prob_pred = torch.softmax(raw_pred, dim=1)
    pred_class = torch.argmax(prob_pred, dim=1).cpu().squeeze().numpy()

# Converting the 0-5 integer masks into visible RGB colors.
rgb_true_mask = color_map[single_mask.astype(int)]
rgb_pred_mask = color_map[pred_class.astype(int)]

# Plotting the results
fig,arr = plt.subplots(1, 3,figsize=(10,8))


# plt.figure(figsize=(10,8))
# plt.subplot(121)
# plt.imshow(image_dataset[0])
# plt.subplot(122)
# plt.imshow(mask_dataset[0])


# Convert original image back to (Height, Width, Channels) for plotting
img_display = single_image[0].cpu().permute(1, 2, 0).numpy()

arr[0].imshow(img_display)
arr[0].set_title("Original Dubai Image")
arr[0].axis('off')

arr[1].imshow(rgb_true_mask)
arr[1].set_title("True Ground Mask")
arr[1].axis('off')

arr[2].imshow(rgb_pred_mask)
arr[2].set_title("Model's Prediction")
arr[2].axis('off')

plt.show()

In [ ]:
import zipfile
import os

zip_path = 'mumbai.zip'
extract_path = './mumbai_data'

print("Unzipping the dataset")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Unzipping complete!")

In [ ]:
import cv2
import torch
import numpy as np
from torch.utils.data import Dataset, DataLoader
import os

class MumbaiDirectoryDataset(Dataset):
    def __init__(self, images_dir, masks_dir):
        self.items = []
        mask_files = os.listdir(masks_dir)
        mask_dict = {os.path.splitext(f)[0]: os.path.join(masks_dir, f) for f in mask_files}

        for img_name in sorted(os.listdir(images_dir)):
            base_name = os.path.splitext(img_name)[0]
            if base_name in mask_dict:
                img_path = os.path.join(images_dir, img_name)
                mask_path = mask_dict[base_name]
                self.items.append((img_path, mask_path))

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img_path, mask_path = self.items[idx]

        # reading image and mask in RGB
        img = cv2.cvtColor(cv2.imread(img_path), cv2.COLOR_BGR2RGB)
        mask_rgb = cv2.cvtColor(cv2.imread(mask_path), cv2.COLOR_BGR2RGB)

        # resize the image and mask accoding to the defined patch size
        img = cv2.resize(img, (256, 256))
        mask_rgb = cv2.resize(mask_rgb, (256, 256), interpolation=cv2.INTER_NEAREST)

        # Translate Mumbai RGB to Dubai Integers (0 to 5)
        integer_mask = np.zeros((256, 256), dtype=np.int64) # Defaults to 0 (Unlabeled)

        # building the classes for land, roads, building etc.
        integer_mask[(mask_rgb == [200, 200, 200]).all(axis=-1)] = 1
        integer_mask[(mask_rgb == [250, 235, 185]).all(axis=-1)] = 1

        integer_mask[(mask_rgb == [200, 160, 40]).all(axis=-1)] = 2

        integer_mask[(mask_rgb == [100, 100, 150]).all(axis=-1)] = 3

        integer_mask[(mask_rgb == [80, 140, 50]).all(axis=-1)] = 4

        integer_mask[(mask_rgb == [40, 120, 240]).all(axis=-1)] = 5

        # converting to Tensors
        img_tensor = torch.tensor(img, dtype=torch.float32).permute(2, 0, 1) / 255.0
        mask_tensor = torch.tensor(integer_mask, dtype=torch.long)

        return img_tensor, mask_tensor

# initializing the loader
mumbai_test_images = './mumbai_data/Dataset/Prepared_Dataset/test/images'
mumbai_test_masks = './mumbai_data/Dataset/Prepared_Dataset/test/masks'
mumbai_test_dataset = MumbaiDirectoryDataset(mumbai_test_images, mumbai_test_masks)
mumbai_test_loader = DataLoader(mumbai_test_dataset, batch_size=16, shuffle=False)

print("Loader is active. Ready for testing")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch


color_map = np.array([   # defining the Dubai hex colors
    [155, 155, 155], # 0: Unlabeled
    [60,  16,  152], # 1: Building
    [132, 41,  246], # 2: Land
    [110, 193, 228], # 3: Road
    [254, 221, 58],  # 4: Vegetation
    [226, 169, 41]   # 5: Water
], dtype=np.uint8)


model.eval()
images, true_masks = next(iter(mumbai_test_loader)) # taking one batch of unseen Mumbai images

# Pick the first image in this batch
idx = 15
single_image = images[idx].unsqueeze(0).to(device)
single_mask = true_masks[idx].numpy()

# Print the raw numbers hiding inside the Mumbai mask
print(f"Unique pixel values in this Mumbai mask: {np.unique(single_mask)}")

# making prediction using Dubai model
with torch.no_grad():
    raw_pred = model(single_image)
    prob_pred = torch.softmax(raw_pred, dim=1)
    pred_class = torch.argmax(prob_pred, dim=1).cpu().squeeze().numpy()

# Colorizing the outputs
rgb_pred_mask = color_map[pred_class.astype(int)] # We are only applying the color to the model's predictions

# Plot the comparison
fig,arr = plt.subplots(1, 3,figsize=(10,8))

img_display = single_image[0].cpu().permute(1, 2, 0).numpy()

arr[0].imshow(img_display)
arr[0].set_title("Unseen Mumbai Image")
arr[0].axis('off')

arr[1].imshow(single_mask, cmap='viridis')
arr[1].set_title("True Mumbai Ground Mask")
arr[1].axis('off')

arr[2].imshow(rgb_pred_mask)
arr[2].set_title("Dubai Model's Prediction")
arr[2].axis('off')

plt.show()

In [ ]:
from google.colab import files
import cv2
import torch
import numpy as np
import matplotlib.pyplot as plt

# 1. Prompt for the Satellite Image
print("--- STEP 1: Upload the Satellite Image (.png, .jpg) ---")
uploaded_img = files.upload()
img_filename = list(uploaded_img.keys())[0]

# 2. Prompt for the Ground Truth Mask
print("\n--- STEP 2: Upload the Corresponding Mask (.png, .tif) ---")
uploaded_mask = files.upload()
mask_filename = list(uploaded_mask.keys())[0]

# 3. Read and format the Satellite Image for the model
img = cv2.imread(img_filename)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
img_resized = cv2.resize(img, (256, 256))

# Convert to tensor and add batch dimension -> (1, 3, 256, 256)
img_tensor = torch.tensor(img_resized, dtype=torch.float32).permute(2, 0, 1).unsqueeze(0) / 255.0
img_tensor = img_tensor.to(device)

# 4. Read the True Mask (Just for visual plotting)
mask = cv2.imread(mask_filename)
mask = cv2.cvtColor(mask, cv2.COLOR_BGR2RGB)
mask_resized = cv2.resize(mask, (256, 256), interpolation=cv2.INTER_NEAREST)

# 5. Run Inference with your Dubai Model
model.eval()
with torch.no_grad():
    raw_pred = model(img_tensor)
    prob_pred = torch.softmax(raw_pred, dim=1)
    pred_class = torch.argmax(prob_pred, dim=1).cpu().squeeze().numpy()

# 6. Colorize the prediction using your specific Dubai Palette
color_map = np.array([
    [155, 155, 155], # 0: Unlabeled
    [60,  16,  152], # 1: Building
    [132, 41,  246], # 2: Land
    [110, 193, 228], # 3: Road
    [254, 221, 58],  # 4: Vegetation
    [226, 169, 41]   # 5: Water
], dtype=np.uint8)

rgb_pred_mask = color_map[pred_class.astype(int)]

# 7. Plot the 1x3 comparison
fig, arr = plt.subplots(1, 3, figsize=(18, 6))

arr[0].imshow(img_resized)
arr[0].set_title("1. Your Uploaded Image")
arr[0].axis('off')

arr[1].imshow(mask_resized)
arr[1].set_title("2. Your Uploaded True Mask")
arr[1].axis('off')

arr[2].imshow(rgb_pred_mask)
arr[2].set_title("3. Dubai Model's Prediction")
arr[2].axis('off')

plt.show()

In [ ]:
import torch
from torchmetrics.classification import MulticlassJaccardIndex
from tqdm.notebook import tqdm # This creates a progress bar in better visualization

# Initialize the 6-class metric
mumbai_iou_metric = MulticlassJaccardIndex(num_classes=6, average='none').to(device)

# putting model in evaluation mode
model.eval()

with torch.no_grad():
    # Wrapping the loader in tqdm gives you a live progress bar!
    for images, masks in tqdm(mumbai_test_loader, desc="Evaluating Batches"):
        images = images.to(device)
        masks = masks.to(device)

        # Forward pass
        raw_predictions = model(images)
        prob_predictions = torch.softmax(raw_predictions, dim=1)
        pred_classes = torch.argmax(prob_predictions, dim=1)

        # Update the metric state with this batch
        mumbai_iou_metric.update(pred_classes, masks)

# 4. Computing the final mathematical IoU across all 891 images using .compute()
mumbai_per_class_iou = mumbai_iou_metric.compute()

class_names = ["Unlabeled", "Building", "Land", "Road", "Vegetation", "Water"]

print("\n\n--- Testing Zero-Shot on MUMBAI")
for name, score in zip(class_names, mumbai_per_class_iou):
    print(f"{name}: {score:.4f}")

print(f"\nOverall Mumbai Average (mIoU): {mumbai_per_class_iou.mean():.4f}")

mumbai_iou_metric.reset()

In [ ]:
from torch.utils.data import random_split, DataLoader

# Defining how many images we want to use for the "Few-Shot" training
num_train_images = 50
num_test_images = len(mumbai_test_dataset) - num_train_images

print(f"Splitting data: {num_train_images} for training, {num_test_images} for testing.")

# 2. Randomly split your existing Mumbai dataset
mumbai_train_subset, mumbai_test_subset = random_split(
    mumbai_test_dataset,
    [num_train_images, num_test_images]
)

# 3. Create a new DataLoader just for these 50 training images
# We use a smaller batch size (e.g., 4 or 8) because the dataset is so small
fewshot_train_loader = DataLoader(mumbai_train_subset, batch_size=8, shuffle=True)

In [ ]:
# 1. Tell PyTorch to stop updating the Encoder's weights
for param in model.encoder.parameters():
    param.requires_grad = False

# 2. Make sure the Decoder is allowed to learn
for param in model.decoder.parameters():
    param.requires_grad = True

# 3. Make sure the final prediction layer is allowed to learn
for param in model.segmentation_head.parameters():
    param.requires_grad = True

print("Encoder frozen! Only the Decoder will learn the new Mumbai features.")

In [ ]:
# The loss functions from your Dubai training (re-initialize them if you disconnected)
from segmentation_models_pytorch.losses import DiceLoss, FocalLoss
dice_fn = DiceLoss(mode='multiclass')
focal_fn = FocalLoss(mode='multiclass')

# CRITICAL: We use 'filter' to only give the optimizer the unfrozen weights
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

epochs = 20 # A quick 15 epochs should be plenty for 50 images

print("Starting Few-Shot Fine-Tuning...")

for epoch in range(epochs):
    model.train()
    train_loss = 0.0

    # Loop through our tiny 50-image dataset
    for images, masks in fewshot_train_loader:
        images = images.to(device)
        masks = masks.to(device)

        optimizer.zero_grad()

        predictions = model(images)
        loss = dice_fn(predictions, masks) + focal_fn(predictions, masks)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    avg_loss = train_loss / len(fewshot_train_loader)
    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {avg_loss:.4f}")

In [ ]:
import torch
from torchmetrics.classification import MulticlassJaccardIndex
from tqdm.notebook import tqdm

# 1. Create a DataLoader for the remaining 841 test images
# We use batch_size=16 again because we aren't training, just predicting
fewshot_test_loader = DataLoader(mumbai_test_subset, batch_size=16, shuffle=False)

print(f"Starting Final Evaluation on {len(mumbai_test_subset)} unseen Mumbai images...")

# 2. Initialize the 6-class metric
final_iou_metric = MulticlassJaccardIndex(num_classes=6, average='none').to(device)

# 3. Lock the model for inference (turns off learning)
model.eval()

# 4. Run the evaluation loop
with torch.no_grad():
    for images, masks in tqdm(fewshot_test_loader, desc="Testing Fine-Tuned Model"):
        images = images.to(device)
        masks = masks.to(device)

        # Forward pass
        raw_predictions = model(images)
        prob_predictions = torch.softmax(raw_predictions, dim=1)
        pred_classes = torch.argmax(prob_predictions, dim=1)

        # Update the score
        final_iou_metric.update(pred_classes, masks)

# 5. Compute the final mathematical IoU
final_per_class_iou = final_iou_metric.compute()

class_names = ["Unlabeled", "Building", "Land", "Road", "Vegetation", "Water"]

print("\n\n--- FEW-SHOT MUMBAI: FINAL RECOVERED IoU SCORES ---")
for name, score in zip(class_names, final_per_class_iou):
    print(f"{name}: {score:.4f}")

print(f"\nOverall Recovered Average (mIoU): {final_per_class_iou.mean():.4f}")

# Clear memory
final_iou_metric.reset()